# Pipeline walkthrough — executed

Every cell below ran against the live platform: real Delta tables on disk, a real
Qdrant collection, and the real retrieval stack.

> **All operational data is synthetic**, generated for training purposes. Zone
> capacities, SLA targets and procedures are illustrative constructions and do
> not represent official figures or procedures of any Saudi authority.

In [1]:
import sys, os
sys.path.insert(0, "/Users/mohammed/codes/hajj-crowd-ops-platform")
os.chdir("/Users/mohammed/codes/hajj-crowd-ops-platform")

import polars as pl
from src.lakehouse import delta_io
pl.Config.set_tbl_width_chars(160)
pl.Config.set_tbl_rows(12)
print("ready")

ready


## 1. The lakehouse is real Delta, not Parquet in a folder

In [2]:
from pathlib import Path
for t in ["bronze_zone_occupancy", "bronze_service_requests",
          "silver_zone_occupancy", "silver_service_requests",
          "gold_zone_hourly", "quarantine"]:
    log = Path(delta_io.table_path(t)) / "_delta_log"
    commits = len(list(log.glob("*.json")))
    print(f"{t:26s} rows={delta_io.read(t).num_rows:>9,}  "
          f"version={delta_io.version(t):>3}  _delta_log commits={commits}")

bronze_zone_occupancy      rows=  185,843  version= 18  _delta_log commits=19
bronze_service_requests    rows=   11,054  version=  1  _delta_log commits=2
silver_zone_occupancy      rows=  185,843  version=  0  _delta_log commits=1
silver_service_requests    rows=    2,500  version=  0  _delta_log commits=1
gold_zone_hourly           rows=    2,688  version=  0  _delta_log commits=1
quarantine                 rows=   14,971  version= 20  _delta_log commits=21


## 2. Bronze carries the Kafka coordinates that produced each row

Any bronze row can be traced back to the exact broker offset it came from.

In [3]:
bronze = pl.from_arrow(delta_io.read("bronze_zone_occupancy"))
print("columns:", bronze.columns)
bronze.select(["event_id", "zone_id", "event_time", "occupancy_estimate",
               "_kafka_topic", "_kafka_partition", "_kafka_offset"]).head(5)

columns: ['event_id', 'zone_id', 'gate_id', 'event_time', 'entries', 'exits', 'occupancy_estimate', 'sensor_status', 'schema_version', '_kafka_topic', '_kafka_partition', '_kafka_offset', '_ingested_at', '_source_file', 'ingest_date']


event_id,zone_id,event_time,occupancy_estimate,_kafka_topic,_kafka_partition,_kafka_offset
str,str,"datetime[μs, UTC]",i32,str,i32,i64
"""91171f55-6682-48a7-8a07-44a755…","""DIRIYAH_TURAIF""",2026-05-30 09:55:41.952 UTC,2855,"""zone_occupancy_raw""",1,68720
"""dfebdd4f-960c-4258-be44-357d44…","""JAM_BRIDGE_L1""",2026-05-30 09:55:41.952 UTC,1113,"""zone_occupancy_raw""",1,68721
"""e4edc919-eda2-4552-8bd8-001a73…","""MATAF_02""",2026-05-30 09:55:41.952 UTC,22745,"""zone_occupancy_raw""",1,68723
"""e7a36f6f-9e84-490a-8b51-1ba5bd…","""ALULA_HEGRA""",2026-05-30 09:56:30.336 UTC,1520,"""zone_occupancy_raw""",1,68724
"""758bebb2-caab-4a23-ac75-21c90e…","""ARAFAT_Z1""",2026-05-30 09:56:30.336 UTC,368,"""zone_occupancy_raw""",1,68725


## 3. The contract rejected malformed records, with reasons

Rejections are queryable, not just logged — the quarantine table sits alongside
the rest of the lakehouse.

In [4]:
q = pl.from_arrow(delta_io.read("quarantine"))
print(f"quarantined records: {q.height:,}\n")
(q.with_columns(pl.col("failed_rules").str.json_decode().alias("rule"))
  .explode("rule")
  .group_by("rule").agg(pl.len().alias("count"))
  .sort("count", descending=True))

quarantined records: 14,971



rule,count
str,u32
"""value_error""",4545
"""literal_error""",1649
"""json_invalid""",1536
"""missing""",1469
"""int_type""",1462
"""greater_than_equal""",1445
"""extra_forbidden""",1442
"""string_type""",1423


In [5]:
# The strict-mode headline: a JSON string refused for an integer field.
coercion = q.filter(pl.col("failed_rules").str.contains("int_type"))
print(f"strict-mode coercion rejections: {coercion.height:,}\n")
print("reason :", coercion["rejection_reason"][0])
print("payload:", coercion["original_payload"][0][:160])

strict-mode coercion rejections: 1,462

reason : entries: Input should be a valid integer
payload: {"event_id": "5c817faa-f8cf-4207-b779-33dc542db477", "zone_id": "MASAA_L1", "gate_id": "G-MASAA_L1-N", "event_time": "2026-05-30T10:15:03.168000Z", "entries": "


## 4. The MERGE produced current state, not an append

One row per `request_id`, holding its latest lifecycle status.

In [6]:
s = pl.from_arrow(delta_io.read("silver_service_requests"))
print(f"rows: {s.height:,}   distinct request_id: {s['request_id'].n_unique():,}")
assert s.height == s["request_id"].n_unique(), "MERGE produced duplicates"
print("uniqueness holds -> this is an upsert, not an append\n")
s.group_by("status").agg(pl.len().alias("n")).sort("n", descending=True)

rows: 2,500   distinct request_id: 2,500
uniqueness holds -> this is an upsert, not an append



status,n
str,u32
"""RESOLVED""",1945
"""CANCELLED""",376
"""ON_SITE""",155
"""DISPATCHED""",12
"""ACKNOWLEDGED""",9
"""REPORTED""",3


## 5. PII governance is observable in the schema

Raw identifiers do not exist in silver; only salted hashes.

In [7]:
print("raw pilgrim_ref present?   ", "pilgrim_ref" in s.columns)
print("raw reporter_phone present?", "reporter_phone" in s.columns)
print("hashed columns:            ", [c for c in s.columns if c.endswith("_hash")])
print("\nexample hash:", s["pilgrim_ref_hash"][0])

raw pilgrim_ref present?    False
raw reporter_phone present? False
hashed columns:             ['pilgrim_ref_hash', 'reporter_phone_hash']

example hash: a7916624fe9b3ae85368b7be08c84640355335fb4b5207239994ecf5a99cc5ee


## 6. Gold is a genuine aggregate

185k readings collapse to one row per zone-hour, with metrics that do not exist
upstream.

In [8]:
silver = pl.from_arrow(delta_io.read("silver_zone_occupancy"))
gold = pl.from_arrow(delta_io.read("gold_zone_hourly"))
print(f"silver rows: {silver.height:,}   gold rows: {gold.height:,}   "
      f"collapse ratio: {silver.height / gold.height:.0f}:1")
print(f"columns only in gold: "
      f"{sorted(set(gold.columns) - set(silver.columns))}")

silver rows: 185,843   gold rows: 2,688   collapse ratio: 69:1
columns only in gold: ['avg_occupancy', 'avg_utilization_pct', 'degraded_sensor_pct', 'minutes_above_80pct', 'minutes_above_90pct', 'peak_occupancy', 'peak_utilization_pct', 'reading_count', 'total_entries', 'total_exits']


In [9]:
# The operational question: how long did each zone spend above its thresholds?
(gold.group_by("zone_id").agg([
    pl.col("minutes_above_80pct").sum().round(1).alias("total_min_above_80"),
    pl.col("minutes_above_90pct").sum().round(1).alias("total_min_above_90"),
    pl.col("peak_utilization_pct").max().alias("max_util_pct"),
]).sort("total_min_above_90", descending=True))

zone_id,total_min_above_80,total_min_above_90,max_util_pct
str,f64,f64,f64
"""MUZ_Z1""",601.8,299.9,98.99
"""MATAF_03""",1073.3,215.2,108.83
"""MASAA_L1""",1044.7,213.7,109.77
"""HARAM_GATE_79""",1026.3,210.8,107.13
"""MASAA_L2""",1074.7,203.6,110.2
"""MATAF_02""",1085.9,198.0,109.43
…,…,…,…
"""JAM_BRIDGE_L2""",362.7,133.3,103.83
"""JAM_BRIDGE_L1""",364.3,122.3,103.93


The shape is real: Haram zones sustain high utilisation across the week,
Jamarat/Arafat/Muzdalifah spike only on their ritual days, and the tourism sites
never approach capacity.

## 7. Hybrid retrieval and RRF

Dense and BM25 rankings fused by hand-written Reciprocal Rank Fusion.

In [10]:
from src.rag.retriever import dense_search, bm25_search, hybrid_search

query = "who signs off on turning people away from a full area"
d = dense_search(query, 10)
b = bm25_search(query, 10)
f, _ = hybrid_search(query, 50)

print(f"query: {query!r}\n")
print(f"{'rank':<5} {'dense':<34} {'bm25':<34} {'RRF fused'}")
for i in range(5):
    print(f"{i+1:<5} {d[i]['payload']['doc_code']:<34} "
          f"{b[i]['payload']['doc_code']:<34} {f[i].doc_code}")

query: 'who signs off on turning people away from a full area'

rank  dense                              bm25                               RRF fused
1     SOP-CS-011                         SOP-CS-015                         SOP-CS-011
2     SOP-SEC-007                        SOP-SEC-007                        SOP-CS-015
3     SOP-CS-011                         FAQ-001                            SOP-SEC-007
4     SOP-CS-015                         SOP-MED-009                        SOP-CS-011
5     SOP-CS-004                         SOP-OPS-001                        SOP-CS-015


In [11]:
# BM25 misses the target document entirely in the top 5; dense carries it.
target = "SOP-CS-011"
for name, hits, get in [("dense", d, lambda h: h["payload"]["doc_code"]),
                        ("bm25 ", b, lambda h: h["payload"]["doc_code"]),
                        ("rrf  ", f, lambda c: c.doc_code)]:
    rank = next((i+1 for i, h in enumerate(hits) if get(h) == target), None)
    print(f"{name}: {target} first appears at rank {rank}")

dense: SOP-CS-011 first appears at rank 1
bm25 : SOP-CS-011 first appears at rank 7
rrf  : SOP-CS-011 first appears at rank 1


## 8. Reranking and a grounded answer with citations

In [12]:
from src.rag.pipeline import ask

answer = ask("What is the response-time SLA for a P1 medical request?")
print("ANSWER:\n")
print(answer.answer)
print("\nMODEL:", answer.model_used)
print("\nCITATIONS:")
for c in answer.citations:
    print(f"  [{c.chunk_number}] {c.doc_code:<14} {c.section[:44]:<46} "
          f"fused={c.fused_score:.6f} rerank={c.rerank_score}")

/Users/mohammed/codes/hajj-crowd-ops-platform/.venv/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


[reranker] loading cross-encoder/ms-marco-MiniLM-L-6-v2 ...


ANSWER:

The response-time SLA for a P1 medical request is **4 minutes** [2, SOP-MED-002] [3, SOP-OPS-001].

MODEL: nvidia/nemotron-3-ultra-550b-a55b:free

CITATIONS:
  [1] SOP-MED-002    2. P1 medical definition                       fused=0.032002 rerank=4.5754
  [2] SOP-MED-002    1. Purpose                                     fused=0.032002 rerank=4.5352
  [3] SOP-OPS-001    3. The SLA matrix                              fused=0.032787 rerank=3.8999
  [4] SOP-MED-002    4. P1 protocol                                 fused=0.030310 rerank=2.8668
  [5] SOP-OPS-001    6. Escalation on breach                        fused=0.029437 rerank=2.0973


In [13]:
# Refusal when the corpus does not cover the question - the behaviour that
# makes a grounded system trustworthy.
print(ask("What is the refund policy for a cancelled Umrah booking?").answer)

This is not covered in the available procedures.


## 9. Lineage was emitted for every stage

In [14]:
import json, collections
counts = collections.Counter()
rows = []
for line in open("docs/evidence/lineage/events.jsonl"):
    e = json.loads(line)
    counts[e["eventType"]] += 1
    for o in e.get("outputs", []):
        st = o.get("outputFacets", {}).get("outputStatistics")
        if st:
            rows.append((e["job"]["name"], o["name"], st["rowCount"]))

print("event types:", dict(counts), "\n")
print("row-count facets on COMPLETE:")
for job, ds, n in rows:
    print(f"  {job:<34} -> {ds:<28} {n:>9,}")

event types: {'START': 8, 'COMPLETE': 8} 

row-count facets on COMPLETE:
  ingest_bronze_zone_occupancy_raw   -> bronze_zone_occupancy          185,843
  ingest_bronze_zone_occupancy_raw   -> quarantine                      14,157
  ingest_bronze_service_requests_raw -> bronze_service_requests         11,054
  ingest_bronze_service_requests_raw -> quarantine                         814
  validate_bronze                    -> ge_validation_bronze           185,843
  build_silver_occupancy             -> silver_zone_occupancy          185,843
  build_silver_requests_merge        -> silver_service_requests          2,500
  validate_silver                    -> ge_validation_silver             2,500
  build_gold_zone_hourly             -> gold_zone_hourly                 2,688
  refresh_rag_index                  -> qdrant://hajj_sop_v1                87


---

Every table, count and answer above was produced by the running platform. See
`docs/RUBRIC_MAP.md` for the full requirement → implementation → evidence map.